In [1]:
from pysat.formula import CNF
from pysat.solvers import Solver
from pysat.solvers import Kissat404
from pysat.solvers import Glucose42
from partitionsolver.utils import file_reader
from multiprocessing import Process, Queue
import partitionsolver.utils.hypergraph_worker as hypergraph_worker
import time
import os
import json

import numpy as np

from partitionsolver.solver.division_solver import DivisionDPLL

import lzma


In [2]:
timings_file = "../output/timings.json"
timings_file_old = "../output/timings_old.json"

#cnf_file_location = "./instances/instances_small/d6afa5689d75e37111656db8980dd54b-grs-160-48.cnf.xz"
#cnf_file_location = "./instances/instances_small/4c3001f8073986116d98084dde70da05-fsf-300-354-2-2-3-2.9.opt.cnf.xz"
#cnf_file_location = "./instances/instances_small/ad9eb96bac59319fc2f7daffd1f961f8-AProVE07-21.cnf.xz"
#cnf_file_location = "./instances/instances_small/77a0d54f2fb3740a9a321623c0c10f3e-tseitin_grid_n12_m12.cnf.xz"
#cnf_file_location = "./instances/instances_small/b628043a07c5576dd6cd21c9d73a69e0-fixedbandwidth-eq-37_shuffled.cnf.xz"
#cnf_file_location = "./instances/custom/two_random_connect8.cnf"
cnf_file_location = "../instances/random_sat/uf50-028.cnf"

In [3]:

stats = {}

def load_stats_file():
    global stats
    if stats is None:
        try:
            with open(timings_file) as f:
                stats = json.load(f)
        except FileNotFoundError:
            stats = {}
    return stats

def save_stats():
    global stats
    if stats is None:
        return
    
    old_stats = {}
    try:
        with open(timings_file_old) as f:
            old_stats = json.load(f)
    except FileNotFoundError:
        old_stats = {}

    with open(timings_file_old, "w") as f:
        json.dump(old_stats, f, indent=2)
    with open(timings_file, "w") as f:
        json.dump(stats, f, indent=2)

def set_stat(path, solver, satisfiable, time, variables, clauses, extradata = None):
    if path not in stats:
        stats[path] = {
            "variables": variables,
            "clauses": clauses
        }
    stats[path][solver] = {
        "satisfiable": satisfiable,
        "time": time,
        "data": extradata
    }

In [ ]:
def extract_common_variables(clausesA, clausesB, num_vars):
    maskA = [False] * num_vars
    maskB = [False] * num_vars
    for clause in clausesA:
        for lit in clause:
            maskA[abs(lit) - 1] = True
    for clause in clausesB:
        for lit in clause:
            maskB[abs(lit) - 1] = True
    
    return [i + 1 for i in range(num_vars) if maskA[i] and maskB[i]]

def create_split_formula(file_location:str, display_progress:bool = False):
    # === Create hypergraph to partition the formula ===
    _, h_clauses, hyperedges = file_reader.hypergraph_from_cnf_xz(file_location, display_progress)
    queue = Queue()
    p = Process(target=hypergraph_worker.create_partition, args=(queue, h_clauses, hyperedges))
    p.start()
    p.join()
    if p.exitcode != 0:
        if display_progress:
            print(f"Partitioning failed with exit code {p.exitcode} ({file_location})")
    del hyperedges
    

    # === Read the clauses from file ===
    num_vars, num_clauses, clauses = file_reader.read_cnf(file_location)
    result = queue.get()
    partition = result["partition"]


    # === Split clauses is two formula, depending on the partition ===
    # todo: Collect clauses that have only glue variables. Add them to both formulas
    clauses_A = [clause.tolist() for clause, p in zip(clauses, partition) if p == 0]
    clauses_B = [clause.tolist() for clause, p in zip(clauses, partition) if p == 1]
    clauses_all = [clause.tolist() for clause in clauses]
    cnf_A = CNF(from_clauses=clauses_A)
    cnf_B = CNF(from_clauses=clauses_B)
    # Some trailing unused variables may get lost - re-add them to the CNF
    cnf_A.nv = num_vars
    cnf_B.nv = num_vars

    glue_variables = extract_common_variables(clauses_A, clauses_B, num_vars)
    #print(f"Clauses: {num_clauses}, variables: {num_vars}, glue_variables: {len(glue_variables)}, length_clauses: {len(clauses)}")
    #print(f"Glue variables: {glue_variables}")

    # === Solve the partial formulas independently ===
    cnf_whole = CNF(from_clauses=clauses_all)
    start = time.perf_counter()
    with Kissat404(bootstrap_with=cnf_whole) as solver:
        sat = solver.solve()
        print(f"Complete solvable: {sat}. Solved in {1000 * (time.perf_counter() - start):.2f}ms")
        set_stat(file_location, "kissat404", sat, (time.perf_counter() - start), num_vars, num_clauses)


    start = time.perf_counter()
    division_solver = DivisionDPLL(num_vars, glue_variables, [clauses_A, clauses_B])
    sat = division_solver.solve()
    set_stat(file_location, "division_solver_jump_forward", sat, (time.perf_counter() - start), num_vars, num_clauses)
    print(f"Division solvable: {sat}. Solved in {1000 * (time.perf_counter() - start):.2f}ms")
    #print(f"Variables: {division_solver.glue_variables}")
    #print(f"Tails: {division_solver.trails}")


#create_split_formula("./instances/random_unsat/uuf50-014.cnf", False)


max = 10
for entry in os.scandir("../instances/random_unsat"):
    max -= 1
    if (max < 0):
        break
    print(f"Next: {entry.name}")
    create_split_formula(entry.path, False)

#save_stats()

Next: uuf50-012.cnf
Complete solvable: False. Solved in 1.98ms
Added initial clauses in 721.29ms
Division solvable: False. Solved in 947.45ms
Next: uuf50-01.cnf
Complete solvable: False. Solved in 0.76ms
Added initial clauses in 744.37ms
Division solvable: False. Solved in 2283.88ms
Next: uuf50-018.cnf
Complete solvable: False. Solved in 0.77ms
Added initial clauses in 968.59ms
Division solvable: False. Solved in 1927.47ms
Next: uuf50-049.cnf
Complete solvable: False. Solved in 2.27ms
Added initial clauses in 1740.70ms
Division solvable: False. Solved in 4987.00ms
Next: uuf50-021.cnf
Complete solvable: False. Solved in 1.22ms
Added initial clauses in 2104.23ms


KeyboardInterrupt: 

In [ ]:
#cnf = CNF(from_clauses=[[-1, 2], [-1, -2]])
cnf = CNF(from_file=cnf_file_location)

# create a SAT solver for this formula:
with Kissat404(bootstrap_with=cnf) as solver:
    print(solver.solve())
    #print(solver)
    #print(solver.get_proof())

False
